In [1]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.hadoop:hadoop-aws:3.3.2,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.180 pyspark-shell"
)

In [2]:
from pyspark.sql import SparkSession
import requests
import time
import json
from datetime import datetime

In [3]:
spark = SparkSession.builder \
    .appName("SparkMinIOExample") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", "Projeto_Final") \
    .config("spark.hadoop.fs.s3a.secret.key", "Projeto_Final") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

In [6]:
def coletar_dados():
    API_TOKEN = "84dfde66637e13d4307f9408fc4d5c36474c7dc322310ef71f264dda3fe75704"
    LOGIN_URL = f"https://api.olhovivo.sptrans.com.br/v2.1/Login/Autenticar?token={API_TOKEN}"

    print("[INFO] Fazendo login na API SPTrans...")
    login_response = requests.post(LOGIN_URL)
    if login_response.status_code != 200:
        raise Exception(f"[ERRO] Falha no login: {login_response.status_code}")

    cookie_value = login_response.cookies.get("apiCredentials")
    if not cookie_value:
        raise Exception("[ERRO] Cookie não retornado pela API")

    print("[INFO] Login bem-sucedido! Capturado cookie de sessão.")

    # ======================================================
    # CLs por região (codigoLinha)
    # ======================================================
    regioes = {
        "zona_norte": [543, 614, 558, 2495, 709],             # Santana, Casa Verde, Vila Maria
        "zona_sul": [1140, 59, 1977, 1318, 1318],             # Santo Amaro, Grajaú, Campo Limpo
        "zona_leste": [2160, 1055, 2580, 1808],               # Itaquera, São Mateus, Aricanduva
        "zona_oeste": [689, 1376, 782, 472],                  # Lapa, Butantã, Pinheiros
        "centro": [1523, 768, 2506, 1366]                     # Sé, República, Liberdade
    }

    registros = []
    ts = time.strftime("%Y-%m-%d %H:%M:%S")

    # ======================================================
    # Coleta de dados por região e CL
    # ======================================================
    for regiao, cls in regioes.items():
        print(f"[INFO] Coletando {len(cls)} linhas da {regiao}...")
        for codigo_linha in cls:
            try:
                url = f"https://api.olhovivo.sptrans.com.br/v2.1/Previsao/Linha?codigoLinha={codigo_linha}"
                resp = requests.get(url, cookies={"apiCredentials": cookie_value})

                if resp.status_code == 200:
                    registros.append({
                        "regiao": regiao,
                        "codigo_linha": codigo_linha,
                        "timestamp": ts,
                        "raw_json": resp.text
                    })
                    print(f"[OK] CL {codigo_linha} ({regiao}) coletado com sucesso.")
                else:
                    registros.append({
                        "regiao": regiao,
                        "codigo_linha": codigo_linha,
                        "timestamp": ts,
                        "raw_json": f"ERRO {resp.status_code}"
                    })
                    print(f"[ERRO] CL {codigo_linha} ({regiao}): {resp.status_code}")

            except Exception as e:
                print(f"[ERRO] Falha ao coletar CL {codigo_linha} ({regiao}): {e}")
                registros.append({
                    "regiao": regiao,
                    "codigo_linha": codigo_linha,
                    "timestamp": ts,
                    "raw_json": f"ERRO: {e}"
                })

    return registros

In [8]:
df = spark.createDataFrame(coletar_dados())

(
    df.write
    .mode("append")
    .partitionBy("regiao", "codigo_linha")
    .json("s3a://bronze/sptrans/previsao/")
)

[INFO] Fazendo login na API SPTrans...
[INFO] Login bem-sucedido! Capturado cookie de sessão.
[INFO] Coletando 5 linhas da zona_norte...
[OK] CL 543 (zona_norte) coletado com sucesso.
[OK] CL 614 (zona_norte) coletado com sucesso.
[OK] CL 558 (zona_norte) coletado com sucesso.
[OK] CL 2495 (zona_norte) coletado com sucesso.
[OK] CL 709 (zona_norte) coletado com sucesso.
[INFO] Coletando 5 linhas da zona_sul...
[OK] CL 1140 (zona_sul) coletado com sucesso.
[OK] CL 59 (zona_sul) coletado com sucesso.
[OK] CL 1977 (zona_sul) coletado com sucesso.
[OK] CL 1318 (zona_sul) coletado com sucesso.
[OK] CL 1318 (zona_sul) coletado com sucesso.
[INFO] Coletando 4 linhas da zona_leste...
[OK] CL 2160 (zona_leste) coletado com sucesso.
[OK] CL 1055 (zona_leste) coletado com sucesso.
[OK] CL 2580 (zona_leste) coletado com sucesso.
[OK] CL 1808 (zona_leste) coletado com sucesso.
[INFO] Coletando 4 linhas da zona_oeste...
[OK] CL 689 (zona_oeste) coletado com sucesso.
[OK] CL 1376 (zona_oeste) coletad